In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from load_data import load_hpo
from analyze_embeddings import (
    load_disease_embeddings,
    compute_nearest_neighbors,
    explain_neighbor_pairs_with_shared_hpo,
    summarize_neighbor_explainability,
    compute_umap_projection,
    save_dataframe,
)

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hpo = load_hpo(RAW_DIR / "hp.obo")

hpoa_filtered = pd.read_csv(PROCESSED_DIR / "hpoa_filtered.csv", dtype=str)

print("Filtered annotation rows:", len(hpoa_filtered))
print("Diseases:", hpoa_filtered["database_id"].nunique())
print("HPO terms:", hpoa_filtered["hpo_id"].nunique())

hpoa_filtered.head()

Filtered annotation rows: 20178
Diseases: 1000
HPO terms: 4569


,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [3]:
embeddings_dp = load_disease_embeddings(PROCESSED_DIR / "disease_embeddings_node2vec_dp.csv")

print("Graph A embeddings:", embeddings_dp.shape)
embeddings_dp.head()

Graph A embeddings: (1000, 35)


,node_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,node_type,label
0,OMIM:619340,-1.299829,-0.655932,0.163469,-0.212847,-0.541585,0.235768,-0.439621,0.706993,-0.094053,...,0.821837,0.886023,0.042071,-0.599112,-0.381867,-0.182142,0.181751,-0.449084,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,-0.326325,-0.137298,-0.193332,-0.442689,0.837968,-0.773861,0.525760,-0.003559,0.135980,...,0.710985,0.379728,-0.270676,-0.485451,0.311005,-0.033072,-0.061557,0.322869,disease,White-Kernohan syndrome
2,OMIM:137580,-0.050143,-0.167317,0.572092,-0.152065,0.805713,-0.183667,-0.217657,0.685667,1.586034,...,0.554641,0.784594,0.590065,-0.747803,-0.186101,-0.170492,-0.984656,-0.651949,disease,Gilles de la tourette syndrome
3,OMIM:108770,-0.231050,-0.012970,-0.462338,-0.735995,0.797530,-0.136100,-0.248430,0.346945,-1.308965,...,-0.796522,1.766711,0.500811,-1.599624,-1.881484,1.273753,0.729441,-1.083026,disease,Atrial standstill 1
4,OMIM:615234,0.262825,-0.077044,-0.354960,0.172508,-0.383768,-0.116940,-0.284227,-0.188844,1.058862,...,-0.234586,1.020362,-0.052442,-0.932727,-0.214134,0.556940,0.323526,0.316028,disease,"Anemia, hypochromic microcytic, with iron over..."


In [4]:
embeddings_dph = load_disease_embeddings(PROCESSED_DIR / "disease_embeddings_node2vec_dph.csv")

print("Graph B embeddings:", embeddings_dph.shape)
embeddings_dph.head()

Graph B embeddings: (1000, 35)


,node_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,node_type,label
0,OMIM:619340,-0.387484,-0.910157,0.982619,0.607956,0.958315,-0.368261,0.394461,0.445716,-0.423625,...,-0.120905,0.763132,-0.503885,-0.006861,-0.187148,0.956883,1.158932,0.808067,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,-0.767676,-0.366015,0.517694,0.434301,0.294414,-0.705663,-0.861528,0.563413,-0.096870,...,0.735737,0.756306,-0.449358,-0.476867,0.487929,-0.073868,0.637030,-0.079910,disease,White-Kernohan syndrome
2,OMIM:137580,0.327134,0.499508,1.228294,-0.119647,1.322186,-1.136861,-0.067582,0.030970,0.003284,...,0.897514,1.406338,0.542065,-0.363319,-0.000300,-0.003450,0.162700,0.031030,disease,Gilles de la tourette syndrome
3,OMIM:108770,0.965642,-1.905361,0.444665,0.971482,1.557175,-0.560559,0.677274,-0.461704,0.438903,...,1.009293,0.851138,-0.061887,0.333160,-0.838361,-0.896541,1.307505,0.096471,disease,Atrial standstill 1
4,OMIM:615234,-0.508136,-0.921777,0.294369,0.567672,1.090082,-0.122706,-0.327817,0.627031,0.638268,...,-0.244192,-0.393572,-1.116120,-0.545966,-1.253565,0.190854,0.288518,-0.912181,disease,"Anemia, hypochromic microcytic, with iron over..."


In [5]:
neighbors_dp = compute_nearest_neighbors(embeddings=embeddings_dp, top_k=10)

print("Neighbor rows:", len(neighbors_dp))
neighbors_dp.head(20)

Neighbor rows: 10000


,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity
0,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:251280,"Microcephaly, seizures, spasticity, and brain ...",1,0.812713
1,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:613722,Developmental and epileptic encephalopathy 12,2,0.768503
2,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620537,Developmental and epileptic encephalopathy 112,3,0.699560
3,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620145,Developmental and epileptic encephalopathy 109,4,0.697447
4,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:619278,"Microcephaly, epilepsy, and diabetes syndrome 2",5,0.693604
5,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620806,Developmental and epileptic encephalopathy 116,6,0.676832
6,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:607745,"Seizures, benign familial infantile, 3",7,0.663000
7,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:621468,Developmental and epileptic encephalopathy 120,8,0.651615
8,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:615476,Developmental and epileptic encephalopathy 18,9,0.641502
9,OMIM:619340,Developmental and epileptic encephalopathy 96,ORPHA:714652,PCDH19 clustering epilepsy,10,0.639160


In [6]:
neighbors_dph = compute_nearest_neighbors(embeddings=embeddings_dph, top_k=10)

print("Neighbor rows:", len(neighbors_dph))
neighbors_dph.head(20)

Neighbor rows: 10000


,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity
0,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:251280,"Microcephaly, seizures, spasticity, and brain ...",1,0.778324
1,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:613722,Developmental and epileptic encephalopathy 12,2,0.761022
2,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620806,Developmental and epileptic encephalopathy 116,3,0.746973
3,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:621468,Developmental and epileptic encephalopathy 120,4,0.732861
4,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620145,Developmental and epileptic encephalopathy 109,5,0.730651
5,OMIM:619340,Developmental and epileptic encephalopathy 96,ORPHA:3006,Pyridoxine-dependent-developmental and epilept...,6,0.729247
6,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:617132,Developmental and epileptic encephalopathy 44,7,0.712811
7,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:220120,D-glyceric aciduria,8,0.686238
8,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:265120,"Surfactant metabolism dysfunction, pulmonary, 1",9,0.678133
9,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620537,Developmental and epileptic encephalopathy 112,10,0.670653


In [7]:
explainability_dp = summarize_neighbor_explainability(neighbors=neighbors_dp, hpoa=hpoa_filtered)

explainability_dp.head(10)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
0,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:251280,"Microcephaly, seizures, spasticity, and brain ...",1,0.812713,3,0.130435,9,17
1,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:613722,Developmental and epileptic encephalopathy 12,2,0.768503,3,0.187500,9,10
2,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620537,Developmental and epileptic encephalopathy 112,3,0.699560,1,0.028571,9,27
3,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620145,Developmental and epileptic encephalopathy 109,4,0.697447,2,0.057143,9,28
4,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:619278,"Microcephaly, epilepsy, and diabetes syndrome 2",5,0.693604,1,0.066667,9,7
5,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620806,Developmental and epileptic encephalopathy 116,6,0.676832,2,0.080000,9,18
6,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:607745,"Seizures, benign familial infantile, 3",7,0.663000,0,0.000000,9,7
7,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:621468,Developmental and epileptic encephalopathy 120,8,0.651615,2,0.042553,9,40
8,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:615476,Developmental and epileptic encephalopathy 18,9,0.641502,2,0.071429,9,21
9,OMIM:619340,Developmental and epileptic encephalopathy 96,ORPHA:714652,PCDH19 clustering epilepsy,10,0.639160,1,0.027027,9,29


In [8]:
explainability_dph = summarize_neighbor_explainability(neighbors=neighbors_dph, hpoa=hpoa_filtered)

explainability_dph.head(10)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
0,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:251280,"Microcephaly, seizures, spasticity, and brain ...",1,0.778324,3,0.130435,9,17
1,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:613722,Developmental and epileptic encephalopathy 12,2,0.761022,3,0.187500,9,10
2,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620806,Developmental and epileptic encephalopathy 116,3,0.746973,2,0.080000,9,18
3,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:621468,Developmental and epileptic encephalopathy 120,4,0.732861,2,0.042553,9,40
4,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620145,Developmental and epileptic encephalopathy 109,5,0.730651,2,0.057143,9,28
5,OMIM:619340,Developmental and epileptic encephalopathy 96,ORPHA:3006,Pyridoxine-dependent-developmental and epilept...,6,0.729247,3,0.073171,9,35
6,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:617132,Developmental and epileptic encephalopathy 44,7,0.712811,1,0.033333,9,22
7,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:220120,D-glyceric aciduria,8,0.686238,1,0.020833,9,40
8,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:265120,"Surfactant metabolism dysfunction, pulmonary, 1",9,0.678133,1,0.043478,9,15
9,OMIM:619340,Developmental and epileptic encephalopathy 96,OMIM:620537,Developmental and epileptic encephalopathy 112,10,0.670653,1,0.028571,9,27


In [9]:
summary_comparison = pd.DataFrame(
    [
        {
            "graph": "disease_phenotype",
            "mean_cosine_similarity_top10": explainability_dp["cosine_similarity"].mean(),
            "mean_shared_hpo_terms_top10": explainability_dp["num_shared_hpo_terms"].mean(),
            "mean_jaccard_hpo_similarity_top10": explainability_dp["jaccard_hpo_similarity"].mean(),
            "pairs_with_zero_shared_hpo_terms": int((explainability_dp["num_shared_hpo_terms"] == 0).sum()),
        },
        {
            "graph": "disease_phenotype_hpo_hierarchy",
            "mean_cosine_similarity_top10": explainability_dph["cosine_similarity"].mean(),
            "mean_shared_hpo_terms_top10": explainability_dph["num_shared_hpo_terms"].mean(),
            "mean_jaccard_hpo_similarity_top10": explainability_dph["jaccard_hpo_similarity"].mean(),
            "pairs_with_zero_shared_hpo_terms": int((explainability_dph["num_shared_hpo_terms"] == 0).sum()),
        },
    ]
)

summary_comparison

,graph,mean_cosine_similarity_top10,mean_shared_hpo_terms_top10,mean_jaccard_hpo_similarity_top10,pairs_with_zero_shared_hpo_terms
0,disease_phenotype,0.681762,3.2878,0.091649,375
1,disease_phenotype_hpo_hierarchy,0.702237,3.2131,0.088294,744


In [10]:
shared_hpo_explanations_dp = explain_neighbor_pairs_with_shared_hpo(neighbors=neighbors_dp, hpoa=hpoa_filtered, hpo=hpo, max_terms_per_pair=20)

shared_hpo_explanations_dp.head(30)

,disease_id,neighbor_id,rank,cosine_similarity,shared_hpo_id,shared_hpo_label,num_shared_hpo_terms,disease_hpo_count,neighbor_hpo_count
0,OMIM:619340,OMIM:251280,1,0.812713,HP:0002187,Profound intellectual disability,3,9,17
1,OMIM:619340,OMIM:251280,1,0.812713,HP:0011451,Primary microcephaly,3,9,17
2,OMIM:619340,OMIM:251280,1,0.812713,HP:0032792,Tonic seizure,3,9,17
3,OMIM:619340,OMIM:613722,2,0.768503,HP:0011097,Epileptic spasm,3,9,10
4,OMIM:619340,OMIM:613722,2,0.768503,HP:0032792,Tonic seizure,3,9,10
5,OMIM:619340,OMIM:613722,2,0.768503,HP:0200134,Epileptic encephalopathy,3,9,10
6,OMIM:619340,OMIM:620537,3,0.699560,HP:0200134,Epileptic encephalopathy,1,9,27
7,OMIM:619340,OMIM:620145,4,0.697447,HP:0011451,Primary microcephaly,2,9,28
8,OMIM:619340,OMIM:620145,4,0.697447,HP:0032792,Tonic seizure,2,9,28
9,OMIM:619340,OMIM:619278,5,0.693604,HP:0001518,Small for gestational age,1,9,7


In [11]:
shared_hpo_explanations_dph = explain_neighbor_pairs_with_shared_hpo(neighbors=neighbors_dph, hpoa=hpoa_filtered, hpo=hpo, max_terms_per_pair=20)

shared_hpo_explanations_dph.head(10)

,disease_id,neighbor_id,rank,cosine_similarity,shared_hpo_id,shared_hpo_label,num_shared_hpo_terms,disease_hpo_count,neighbor_hpo_count
0,OMIM:619340,OMIM:251280,1,0.778324,HP:0002187,Profound intellectual disability,3,9,17
1,OMIM:619340,OMIM:251280,1,0.778324,HP:0011451,Primary microcephaly,3,9,17
2,OMIM:619340,OMIM:251280,1,0.778324,HP:0032792,Tonic seizure,3,9,17
3,OMIM:619340,OMIM:613722,2,0.761022,HP:0011097,Epileptic spasm,3,9,10
4,OMIM:619340,OMIM:613722,2,0.761022,HP:0032792,Tonic seizure,3,9,10
5,OMIM:619340,OMIM:613722,2,0.761022,HP:0200134,Epileptic encephalopathy,3,9,10
6,OMIM:619340,OMIM:620806,3,0.746973,HP:0032792,Tonic seizure,2,9,18
7,OMIM:619340,OMIM:620806,3,0.746973,HP:0200134,Epileptic encephalopathy,2,9,18
8,OMIM:619340,OMIM:621468,4,0.732861,HP:0011097,Epileptic spasm,2,9,40
9,OMIM:619340,OMIM:621468,4,0.732861,HP:0032792,Tonic seizure,2,9,40


In [12]:
# high embedding similarity but zero shared HPO terms

interesting_dph = explainability_dph[explainability_dph["num_shared_hpo_terms"] == 0].sort_values(by="cosine_similarity", ascending=False)

interesting_dph.head(10)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
3126,OMIM:602093,Cone dystrophy 3,OMIM:616151,"Macular dystrophy, vitelliform, 4",7,0.820890,0,0.0,5,5
3352,OMIM:616151,"Macular dystrophy, vitelliform, 4",OMIM:602093,Cone dystrophy 3,3,0.820890,0,0.0,5,5
6333,OMIM:618697,Retinitis pigmentosa 87 with choroidal involve...,OMIM:610478,Retinal cone dystrophy 4,4,0.813846,0,0.0,5,7
4845,OMIM:610478,Retinal cone dystrophy 4,OMIM:618697,Retinitis pigmentosa 87 with choroidal involve...,6,0.813846,0,0.0,7,5
7551,ORPHA:401815,Autosomal recessive spastic paraplegia type 66,OMIM:607584,"Spastic paraplegia 24, autosomal recessive",2,0.805802,0,0.0,14,6
297,OMIM:607584,"Spastic paraplegia 24, autosomal recessive",ORPHA:401815,Autosomal recessive spastic paraplegia type 66,8,0.805802,0,0.0,6,14
5691,OMIM:616736,"Tremor, hereditary essential, 5",OMIM:615957,Spinocerebellar ataxia 38,2,0.801754,0,0.0,5,12
5185,OMIM:615957,Spinocerebellar ataxia 38,OMIM:616736,"Tremor, hereditary essential, 5",6,0.801754,0,0.0,12,5
3353,OMIM:616151,"Macular dystrophy, vitelliform, 4",OMIM:603075,"Macular degeneration, age-related, 1",4,0.801274,0,0.0,5,7
5941,OMIM:603075,"Macular degeneration, age-related, 1",OMIM:616151,"Macular dystrophy, vitelliform, 4",2,0.801274,0,0.0,7,5


In [13]:
easy_case_dph = explainability_dph[explainability_dph["num_shared_hpo_terms"] >= 5].sort_values(by=["num_shared_hpo_terms", "cosine_similarity"], ascending=False)

easy_case_dph.head(20)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
480,OMIM:607625,"Niemann-pick disease, type C2",OMIM:257220,"Niemann-pick disease, type C1",1,0.931260,25,0.657895,33,30
1060,OMIM:257220,"Niemann-pick disease, type C1",OMIM:607625,"Niemann-pick disease, type C2",1,0.931260,25,0.657895,30,33
2640,OMIM:233710,"Granulomatous disease, chronic, autosomal rece...",OMIM:306400,"Chronic granulomatous disease, X-linked",1,0.930677,23,0.522727,36,31
4040,OMIM:306400,"Chronic granulomatous disease, X-linked",OMIM:233710,"Granulomatous disease, chronic, autosomal rece...",1,0.930677,23,0.522727,31,36
4860,OMIM:616708,Desanto-Shinawi syndrome,OMIM:618872,Nizon-Isidor syndrome,1,0.890331,17,0.265625,37,44
6360,OMIM:618872,Nizon-Isidor syndrome,OMIM:616708,Desanto-Shinawi syndrome,1,0.890331,17,0.265625,44,37
520,OMIM:600376,"Telangiectasia, hereditary hemorrhagic, type 2",OMIM:610655,"Telangiectasia, hereditary hemorrhagic, type 4",1,0.882430,17,0.395349,38,22
3630,OMIM:610655,"Telangiectasia, hereditary hemorrhagic, type 4",OMIM:600376,"Telangiectasia, hereditary hemorrhagic, type 2",1,0.882430,17,0.395349,22,38
521,OMIM:600376,"Telangiectasia, hereditary hemorrhagic, type 2",ORPHA:774,Hereditary hemorrhagic telangiectasia,2,0.872411,17,0.298246,38,36
8470,ORPHA:774,Hereditary hemorrhagic telangiectasia,OMIM:600376,"Telangiectasia, hereditary hemorrhagic, type 2",1,0.872411,17,0.298246,36,38


In [14]:
projection_dp = compute_umap_projection(embeddings=embeddings_dp, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=5)

projection_dp.head()

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,node_id,x,y,node_type,label
0,OMIM:619340,7.899555,1.465667,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,6.748061,3.102769,disease,White-Kernohan syndrome
2,OMIM:137580,7.832777,2.121331,disease,Gilles de la tourette syndrome
3,OMIM:108770,4.315359,-1.636832,disease,Atrial standstill 1
4,OMIM:615234,4.609743,0.291094,disease,"Anemia, hypochromic microcytic, with iron over..."


In [15]:
projection_dph = compute_umap_projection(embeddings=embeddings_dph, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=5)

projection_dph.head()

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,node_id,x,y,node_type,label
0,OMIM:619340,9.229444,4.989959,disease,Developmental and epileptic encephalopathy 96
1,OMIM:619426,8.075353,6.995924,disease,White-Kernohan syndrome
2,OMIM:137580,9.055515,6.169488,disease,Gilles de la tourette syndrome
3,OMIM:108770,4.788445,1.810691,disease,Atrial standstill 1
4,OMIM:615234,4.211440,5.258825,disease,"Anemia, hypochromic microcytic, with iron over..."


In [16]:
fig = px.scatter(
    projection_dp,
    x="x",
    y="y",
    hover_name="label",
    hover_data=["node_id"],
    title="Node2Vec disease embeddings: disease-phenotype graph"
)

fig.show()

In [17]:
fig = px.scatter(
    projection_dph,
    x="x",
    y="y",
    hover_name="label",
    hover_data=["node_id"],
    title="Node2Vec disease embeddings: disease-phenotype + HPO hierarchy graph"
)

fig.show()

In [18]:
save_dataframe(
    neighbors_dp,
    PROCESSED_DIR / "neighbors_node2vec_dp.csv",
)

save_dataframe(
    neighbors_dph,
    PROCESSED_DIR / "neighbors_node2vec_dph.csv",
)

save_dataframe(
    explainability_dp,
    PROCESSED_DIR / "neighbor_explainability_node2vec_dp.csv",
)

save_dataframe(
    explainability_dph,
    PROCESSED_DIR / "neighbor_explainability_node2vec_dph.csv",
)

save_dataframe(
    shared_hpo_explanations_dp,
    PROCESSED_DIR / "shared_hpo_explanations_node2vec_dp.csv",
)

save_dataframe(
    shared_hpo_explanations_dph,
    PROCESSED_DIR / "shared_hpo_explanations_node2vec_dph.csv",
)

save_dataframe(
    projection_dp,
    PROCESSED_DIR / "projection_umap_node2vec_dp.csv",
)

save_dataframe(
    projection_dph,
    PROCESSED_DIR / "projection_umap_node2vec_dph.csv",
)

save_dataframe(
    summary_comparison,
    PROCESSED_DIR / "embedding_comparison_summary.csv",
)